# Project PUENTE - VS Code Tunnel Setup (Google Colab)

Run Cells 1 to 3 in order inside a fresh Google Colab browser runtime.
Cell 3 is long-running and prints the one-time GitHub authentication URL and code.

Important: do not open the connectivity probe endpoint in Cell 3. It can return HTTP 404 by design.
Only open the GitHub device login URL that appears after the tunnel starts.

If you run this notebook in a local VS Code kernel, it will fail by design.
Open this notebook in Colab (colab.research.google.com) when you need the tunnel workflow.

Drive mount is optional.
- If your project is in Drive, keep mount enabled and use /content/drive/MyDrive/ProjectPuenteCloud.
- If your project is not in Drive, skip mount and use /content/ProjectPuente (ephemeral).

In [ ]:
# Cell 1: Optional Google Drive mount (for persistent artifacts)
import importlib
import os

try:
    colab_mod = importlib.import_module('google.colab')
except ModuleNotFoundError as exc:
    raise RuntimeError(
        'This notebook must run in Google Colab browser runtime. Open it in colab.research.google.com.'
    ) from exc

mount_drive = os.getenv('PUENTE_MOUNT_DRIVE', 'true').strip().casefold() in {'1', 'true', 'yes', 'y', 'on'}
if mount_drive:
    colab_mod.drive.mount('/content/drive')
else:
    print('Skipping Drive mount (PUENTE_MOUNT_DRIVE=false).')
    print('Use /content/ProjectPuente as your project root for this session.')

ModuleNotFoundError: No module named 'google'

In [ ]:
%%bash
# Cell 2: Install tunnel dependencies and VS Code CLI
set -euo pipefail

apt-get update -y
apt-get install -y curl tar

mkdir -p /content/vscode-cli
curl -fsSL "https://code.visualstudio.com/sha/download?build=stable&os=cli-alpine-x64" -o /content/vscode_cli.tar.gz
tar -xzf /content/vscode_cli.tar.gz -C /content/vscode-cli

# Ensure CLI is executable
chmod +x /content/vscode-cli/code

echo "VS Code CLI installed at /content/vscode-cli/code"

In [ ]:
# Cell 3: Authenticate, verify login state, then start secure VS Code tunnel (long-running by design)
import os
import subprocess
import time
from urllib.parse import urlparse

CODE_BIN = "/content/vscode-cli/code"
CLI_DATA_DIR = "/content/vscode-cli-data"
TUNNEL_NAME = "puente-colab-rde"
FORCE_FRESH_DEVICE_LOGIN = True  # Set to False after your first successful connection.
REQUIRED_URLS = [
    "https://github.com",
    "https://global.rel.tunnels.api.visualstudio.com",
]

if not os.path.exists(CODE_BIN):
    raise FileNotFoundError(
        "Missing /content/vscode-cli/code. Re-run Cell 2 first."
    )

os.makedirs(CLI_DATA_DIR, exist_ok=True)

# Force plain-text CLI output so device-code lines are visible in notebook output.
env = os.environ.copy()
env["TERM"] = "dumb"
env["VSCODE_CLI_DATA_DIR"] = CLI_DATA_DIR

def _probe_url(url: str) -> tuple[bool, str]:
    probe = subprocess.run(
        [
            "curl",
            "-sS",
            "-o",
            "/dev/null",
            "-w",
            "%{http_code}",
            "--connect-timeout",
            "10",
            "--max-time",
            "20",
            url,
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    http_code = (probe.stdout or "").strip() or "000"
    reachable = probe.returncode == 0 and http_code != "000"
    return reachable, http_code

def _show_logged_in_user() -> tuple[bool, str]:
    result = subprocess.run(
        [CODE_BIN, "tunnel", "user", "show"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
        env=env,
    )
    output = (result.stdout or "").strip()
    return result.returncode == 0, (output or "(no output)")

print("Checking required outbound connectivity...")
for url in REQUIRED_URLS:
    host = urlparse(url).netloc or url
    reachable, http_code = _probe_url(url)
    status = "OK" if reachable else "FAIL"
    print(f"[{status}] {host} (HTTP {http_code})")
    if not reachable:
        raise RuntimeError(
            f"Cannot reach {host}. Tunnel auth/link will not appear until outbound network access is available."
        )

print("CLI version:")
subprocess.run([CODE_BIN, "--version"], check=True, env=env)

if FORCE_FRESH_DEVICE_LOGIN:
    print("\nForcing a fresh device-login flow...")
    subprocess.run([CODE_BIN, "tunnel", "user", "logout"], check=False, env=env)
    time.sleep(1)

logged_in, account_text = _show_logged_in_user()
if logged_in and not FORCE_FRESH_DEVICE_LOGIN:
    print(f"Already logged in to tunnel service: {account_text}")
else:
    print("\nStep 1/2: GitHub device login required")
    print("Open only: https://github.com/login/device")
    print("Enter the code printed by the CLI below, then wait for this cell to continue.")
    login_result = subprocess.run(
        [
            CODE_BIN,
            "tunnel",
            "user",
            "login",
            "--provider",
            "github",
            "--verbose",
            "--log",
            "trace",
        ],
        check=False,
        env=env,
    )
    if login_result.returncode != 0:
        raise RuntimeError(
            "GitHub device login did not complete. Do not interrupt early; re-run Cell 3 and finish device activation."
        )

    logged_in, account_text = _show_logged_in_user()
    if not logged_in:
        raise RuntimeError(
            "Device activation page may have succeeded, but tunnel login was not finalized yet. Wait 5-10 seconds and re-run Cell 3."
        )
    print(f"Tunnel login confirmed: {account_text}")

print(f"\nStep 2/2: Starting tunnel '{TUNNEL_NAME}'...")
print("After the tunnel starts, keep this cell running.")
print("On local VS Code: Remote Tunnels: Connect to Tunnel -> select this machine.")

# Clean up any prior tunnel process tied to this CLI data dir/machine.
subprocess.run([CODE_BIN, "tunnel", "kill"], check=False, env=env)
time.sleep(1)

subprocess.run(
    [
        CODE_BIN,
        "tunnel",
        "--accept-server-license-terms",
        "--name",
        TUNNEL_NAME,
        "--verbose",
        "--log",
        "trace",
    ],
    check=True,
    env=env,
 )

## Optional Keep-Alive Browser Snippet (Paste in Browser Console)

This client-side helper clicks the Colab connect control every 10 minutes to reduce idle disconnect risk.

```javascript
(() => {
  const clickConnect = () => {
    const button =
      document.querySelector('colab-connect-button')?.shadowRoot?.querySelector('#connect') ||
      document.querySelector('colab-toolbar-button#connect') ||
      document.querySelector('paper-button#connect');

    if (button) {
      button.click();
      console.log('[puente-keepalive] connect clicked at', new Date().toISOString());
    } else {
      console.log('[puente-keepalive] connect button not found');
    }
  };

  clickConnect();
  window.puenteKeepAliveInterval = setInterval(clickConnect, 10 * 60 * 1000);
})();
```